# 05 — Tổng hợp đánh giá & Khuyến nghị

Mục tiêu:
- Tổng hợp kết quả từ tất cả phương pháp
- So sánh bảng + biểu đồ
- Đưa ra insight và khuyến nghị hành động
- Kết luận và hướng phát triển

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

from src.data.loader import load_data, load_config
from src.data.cleaner import HRDataCleaner
from src.features.builder import FeatureBuilder
from src.models.supervised import train_random_forest, train_xgboost
from src.models.semi_supervised import mask_labels, train_semi_supervised
from src.models.regression import train_satisfaction_regressor, check_leakage
from src.evaluation.metrics import evaluate_classifier, explain_with_shap, build_comparison_table
from src.visualization.plots import plot_model_comparison, plot_feature_importance_top10

config = load_config("configs/params.yaml")

## 5.1 Chuẩn bị dữ liệu

In [ ]:
df = load_data(config["data"]["raw_path"])
cleaner = HRDataCleaner(target_col="Attrition")
df_clean = cleaner.clean(df)
fb = FeatureBuilder(df_clean)
df_featured = fb.build_all()
df_encoded = cleaner.encode(df_featured)

X = df_encoded.drop(columns=["Attrition"])
y = df_encoded["Attrition"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## 5.2 Huấn luyện & Đánh giá tất cả mô hình

In [ ]:
# Random Forest
rf = train_random_forest(X_train, y_train, n_estimators=100, random_state=42)
rf_r = evaluate_classifier(y_test, rf.predict(X_test), rf.predict_proba(X_test)[:,1], "Random Forest")
rf_r["model_name"] = "Random Forest"

# XGBoost
xgb = train_xgboost(X_train, y_train, n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42)
xgb_r = evaluate_classifier(y_test, xgb.predict(X_test), xgb.predict_proba(X_test)[:,1], "XGBoost")
xgb_r["model_name"] = "XGBoost"

# Self-Training (20% labels)
y_semi = mask_labels(y_train.values, labeled_ratio=0.20, random_state=42)
semi = train_semi_supervised(X_train.values, y_semi, n_estimators=100, random_state=42)
semi_r = evaluate_classifier(y_test, semi.predict(X_test.values), semi.predict_proba(X_test.values)[:,1], "Self-Training (20%)")
semi_r["model_name"] = "Self-Training (20%)"

## 5.3 Bảng tổng hợp so sánh mô hình

In [ ]:
comparison = build_comparison_table([rf_r, xgb_r, semi_r])
plot_model_comparison(comparison)
comparison

## 5.4 Top 10 Features (Global Summary)

In [ ]:
shap_values, feature_imp = explain_with_shap(rf, X_test)
if feature_imp is not None:
    plot_feature_importance_top10(feature_imp)
    feature_imp

## 5.5 Hồi quy mức độ hài lòng

In [ ]:
leakage = check_leakage(df_encoded, "JobSatisfaction")
reg_result = train_satisfaction_regressor(
    df_encoded, target_col="JobSatisfaction",
    drop_cols=leakage["leaked_features"],
)
if reg_result:
    reg_result["results"]

## 5.6 Insight → Hành động (Business Recommendations)

### 🎯 Khuyến nghị chính sách HR dựa trên kết quả phân tích

| # | Phát hiện | Insight | Đề xuất hành động |
|---|---|---|---|
| 1 | OT → nghỉ việc (Lift cao) | NV làm thêm giờ có rủi ro nghỉ việc cao gấp 3x | **Giảm OT bắt buộc**, thưởng OT công bằng |
| 2 | Travel thường xuyên | Đi công tác nhiều gây kiệt sức | **Chính sách hybrid/remote** |
| 3 | Thu nhập thấp | Lương không cạnh tranh | **Điều chỉnh lương** theo market rate |
| 4 | NV trẻ (< 30) | Career path chưa rõ | **Chương trình mentor**, lộ trình thăng tiến |
| 5 | Cụm rủi ro cao | Attrition rate > 25% | **Ưu tiên can thiệp**: 1-1 meeting |
| 6 | Single nghỉ nhiều | Ít ràng buộc, dễ chuyển việc | **Môi trường làm việc hấp dẫn** |

### Lưu ý khi áp dụng bán giám sát:
- Pseudo-label có rủi ro False Positive → tốn chi phí can thiệp không cần thiết
- Nên **kết hợp mô hình + kiểm tra thủ công** cho các trường hợp gần ngưỡng
- Khi thu thập >= 30% nhãn, Self-Training cho kết quả đáng tin cậy

## 5.7 Kết luận & Hướng phát triển

### Kết luận

1. **Luật kết hợp**: Xác định tổ hợp yếu tố chính dẫn đến nghỉ việc (OT + Low Satisfaction + Young)
2. **Phân cụm**: 3 nhóm NV với chiến lược HR khác nhau
3. **Phân lớp**: XGBoost/RF đạt F1 và PR-AUC cao trên dữ liệu mất cân bằng
4. **Bán giám sát**: Self-Training cải thiện đáng kể khi chỉ có 5-10% nhãn
5. **Hồi quy**: Dự đoán mức hài lòng với MAE chấp nhận được

### Hướng phát triển

- Thử thêm LightGBM, CatBoost
- Áp dụng Label Propagation thay vì chỉ Self-Training
- Tích hợp temporal analysis nếu có dữ liệu theo thời gian
- Phát triển dashboard real-time cho HR team
- A/B testing chính sách can thiệp trên nhóm rủi ro cao